In [2]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------
# Q1. Load data and print initial diagnostics
# ---------------------------------------------------------
df = pd.read_csv("/content/meridian_daily_sales_raw - meridian_daily_sales_raw.csv.csv")
print("Initial Shape:", df.shape)
print("Initial Types:\n", df.dtypes)

Initial Shape: (1627, 10)
Initial Types:
 store_id             object
store_name           object
region               object
date                 object
transactions          int64
revenue              object
avg_basket_value    float64
discount_pct         object
footfall              int64
stockout_flag        object
dtype: object


In [4]:
for col in ['store_id', 'store_name', 'region']:
    df[col] = df[col].astype(str).str.strip()

df['store_id'] = df['store_id'].str.upper()
df['region'] = df['region'].str.title()
df['store_name'] = df['store_name'].str.title()

In [5]:
df['date'] = pd.to_datetime(df['date'], format='mixed', dayfirst=False)

In [7]:
print("Missing values per column:\n", df.isnull().sum())

Missing values per column:
 store_id             0
store_name           0
region               0
date                 0
transactions         0
revenue             53
avg_basket_value     0
discount_pct         0
footfall             0
stockout_flag        0
dtype: int64


In [9]:
def clean_revenue(val):
    if pd.isna(val) or str(val).strip().lower() in ['', 'nan', 'none', 'na']:
        return np.nan
    val_str = str(val).replace('$', '').replace(',', '').strip()
    # Handle currency conversion if text contains INR (assuming 1 USD = 83 INR for standardization if needed)
    # If simply stripping text wrappers:
    if 'INR' in val_str:
        val_str = val_str.replace('INR', '').strip()
        # Direct conversion if treating INR as raw numeric needing USD mapping:
        # return float(val_str) / 83.0
    return float(val_str)

df['revenue'] = df['revenue'].apply(clean_revenue)
df['revenue'] = df.groupby('region')['revenue'].transform(lambda x: x.fillna(x.median()))

In [10]:
# ---------------------------------------------------------
# Q5. Standardize stockout_flag to boolean
# ---------------------------------------------------------
bool_map = {'Y': True, '1': True, 'TRUE': True, 'YES': True, 'N': False, '0': False, 'FALSE': False, 'NO': False}
df['stockout_flag'] = df['stockout_flag'].astype(str).str.upper().str.strip().map(bool_map)

In [11]:
# ---------------------------------------------------------
# Q6. Remove duplicates (casing standardized prior makes this exact)
# ---------------------------------------------------------
rows_before_dedup = len(df)
df = df.drop_duplicates(subset=['store_id', 'date'], keep='first')
print(f"Deduplication complete. Rows before: {rows_before_dedup}, Rows after: {len(df)}")

Deduplication complete. Rows before: 1627, Rows after: 1608


In [12]:
# ---------------------------------------------------------
# Q7. Handle transactional & discount errors
# Strategy: Cap/Filter negative transactions and discount invalid entries
# ---------------------------------------------------------
# Transactions < 0 are refund glitches -> set to absolute or NaN (imputed)
df['transactions'] = df['transactions'].apply(lambda x: abs(x) if pd.notna(x) and x < 0 else x)

# Discount_pct outside [0, 100] -> set invalid entries (>100 or <0) to NaN and impute median
df['discount_pct'] = pd.to_numeric(df['discount_pct'], errors='coerce')
df.loc[(df['discount_pct'] < 0) | (df['discount_pct'] > 100), 'discount_pct'] = np.nan
df['discount_pct'] = df['discount_pct'].fillna(df['discount_pct'].median())

In [13]:
# ---------------------------------------------------------
# Q8. Save cleaned CSV and report row count
# ---------------------------------------------------------
df.to_csv("meridian_daily_sales_cleaned.csv", index=False)
print(f"Cleaned dataset saved. Final Row Count: {len(df)}")

Cleaned dataset saved. Final Row Count: 1608


In [14]:
# ---------------------------------------------------------
# Q9–Q11. Quartiles, IQR, and Outliers for Revenue
# ---------------------------------------------------------
Q1_rev = df['revenue'].quantile(0.25)
Q2_rev = df['revenue'].median()
Q3_rev = df['revenue'].quantile(0.75)
IQR_rev = Q3_rev - Q1_rev

lower_fence_rev = Q1_rev - 1.5 * IQR_rev
upper_fence_rev = Q3_rev + 1.5 * IQR_rev

rev_outliers = df[(df['revenue'] < lower_fence_rev) | (df['revenue'] > upper_fence_rev)]
print(f"Revenue Fences: [{lower_fence_rev:.2f}, {upper_fence_rev:.2f}]")
print(f"Revenue Outlier Count: {len(rev_outliers)}")

Revenue Fences: [-1125.07, 11581.06]
Revenue Outlier Count: 70


In [15]:
# ---------------------------------------------------------
# Q12. Footfall Outlier Detection
# ---------------------------------------------------------
Q1_ff = df['footfall'].quantile(0.25)
Q3_ff = df['footfall'].quantile(0.75)
IQR_ff = Q3_ff - Q1_ff
upper_fence_ff = Q3_ff + 1.5 * IQR_ff

ff_outliers = df[df['footfall'] > upper_fence_ff]
print(f"Footfall Upper Fence: {upper_fence_ff:.2f}")
print(f"Footfall Outliers caught (including ~40k values): {len(ff_outliers)}")

Footfall Upper Fence: 1008.88
Footfall Outliers caught (including ~40k values): 6


In [16]:
# ---------------------------------------------------------
# Q15. Recompute Footfall Fences without Sensor Faults (< 5000)
# ---------------------------------------------------------
df_clean_ff = df[df['footfall'] < 5000]
Q1_ff_c = df_clean_ff['footfall'].quantile(0.25)
Q3_ff_c = df_clean_ff['footfall'].quantile(0.75)
IQR_ff_c = Q3_ff_c - Q1_ff_c
upper_fence_ff_c = Q3_ff_c + 1.5 * IQR_ff_c

print(f"Original Footfall Upper Fence: {upper_fence_ff:.2f}")
print(f"Cleaned Footfall Upper Fence: {upper_fence_ff_c:.2f}")

Original Footfall Upper Fence: 1008.88
Cleaned Footfall Upper Fence: 1007.50


In [17]:
# ---------------------------------------------------------
# Q17–Q18, Q21. Skewness Analysis
# ---------------------------------------------------------
mean_rev = df['revenue'].mean()
median_rev = df['revenue'].median()
mode_rev = df['revenue'].mode()[0]
skew_rev = df['revenue'].skew()

print(f"Revenue - Mean: {mean_rev:.2f}, Median: {median_rev:.2f}, Mode: {mode_rev:.2f}")
print(f"Revenue Skewness: {skew_rev:.2f}")

skew_ff_raw = df['footfall'].skew()
skew_ff_clean = df_clean_ff['footfall'].skew()
print(f"Footfall Skew (Raw): {skew_ff_raw:.2f} -> Cleaned: {skew_ff_clean:.2f}")

Revenue - Mean: 5588.00, Median: 4876.28, Mode: 4287.35
Revenue Skewness: 1.43
Footfall Skew (Raw): 22.87 -> Cleaned: 0.03


In [18]:
# ---------------------------------------------------------
# Q23. Regional Breakdown Table
# ---------------------------------------------------------
regional_summary = df.groupby('region').apply(lambda g: pd.Series({
    'Median Revenue': g['revenue'].median(),
    'IQR Revenue': g['revenue'].quantile(0.75) - g['revenue'].quantile(0.25),
    'Outlier Count': len(g[(g['revenue'] < (g['revenue'].quantile(0.25) - 1.5*(g['revenue'].quantile(0.75)-g['revenue'].quantile(0.25)))) |
                          (g['revenue'] > (g['revenue'].quantile(0.75) + 1.5*(g['revenue'].quantile(0.75)-g['revenue'].quantile(0.25))))])
})).reset_index()

print("\nRegional Summary:")
print(regional_summary)


Regional Summary:
  region  Median Revenue  IQR Revenue  Outlier Count
0  North        4861.090    3034.1525           19.0
1  South        5695.995    3703.4125           23.0
2   West        4287.350    2678.7150           25.0


/tmp/ipykernel_9132/3008995070.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  regional_summary = df.groupby('region').apply(lambda g: pd.Series({
